# PCES Benchmark Demo
**Google DeepMind Hackathon | Taipan + Will**

In [ ]:
import sys, os, json, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '..')
from pce_score import build_graph, score_coherence, generate_dataset, run_evaluation
from meta_b import predict_difficulty, meta_score, run_meta_evaluation
from rule_switch import surprise_curve, simulate_agent, run_switch_evaluation
from social_phi import phi_iit, run_social_evaluation
from transfer import iso_variant, transfer_score, run_transfer_evaluation
from taipan_eval import taipan_eval, run_taipan_evaluation
print('Imports OK')

In [ ]:
np.random.seed(42)
import random; random.seed(42)
os.makedirs('/tmp/demo', exist_ok=True)
print('Semilla fija OK')

In [ ]:
import hashlib
adj4, r4 = build_graph(hashlib.sha256(b'demo4').hexdigest()[:16], 4, 3)
adj8, r8 = build_graph(hashlib.sha256(b'demo8').hexdigest()[:16], 8, 4)
print('Grafo 4 nodos:', adj4.shape, '| edges:', int(adj4.sum()//2))
print('Grafo 8 nodos:', adj8.shape, '| edges:', int(adj8.sum()//2))
np.savetxt('/tmp/demo/adj4.csv', adj4, delimiter=',')
np.savetxt('/tmp/demo/adj8.csv', adj8, delimiter=',')
print('Grafos guardados')

In [ ]:
generate_dataset(50, '/tmp/demo/dataset.jsonl')
resultado_a = run_evaluation('/tmp/demo/dataset.jsonl')
print('(a) Coherencia baseline:', resultado_a['score_promedio'])
resultado_b = run_meta_evaluation('/tmp/demo/dataset.jsonl')
print('(b) Brier baseline:', resultado_b['brier_medio'])

In [ ]:
score4 = taipan_eval('a', adj=adj4, true_switch=1)
score8 = taipan_eval('a', adj=adj8, true_switch=1)
print('PCES score 4-nodo =', score4)
print('PCES score 8-nodo =', score8)

In [ ]:
actions = simulate_agent(100, 4, 50, seed=42)
kl_max, recovery = surprise_curve(actions, 50)
print('Cambio detectado | KL max:', kl_max, '| Recovery steps:', recovery)
print('Cambio real: paso 50')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
deg4 = adj4.sum(axis=0)
deg8 = adj8.sum(axis=0)
ax1.bar(range(len(deg4)), deg4, label='4-node graph', color='steelblue')
ax1.bar(range(len(deg4), len(deg4)+len(deg8)), deg8, label='8-node graph', color='coral')
ax1.set_title('Degree Distribution')
ax1.legend()
actions_arr = np.array(actions)
ax2.plot(actions_arr, alpha=0.7, color='green')
ax2.axvline(x=50, color='red', linestyle='--', label='Rule Switch')
ax2.set_title('Action sequence with rule switch at step 50')
ax2.legend()
plt.tight_layout()
plt.savefig('/tmp/demo/figura_pces.png', dpi=150)
plt.show()
print('Figura guardada')

In [ ]:
resultado_final = run_taipan_evaluation()
print('SCORES FINALES TAIPAN:')
for comp, vs in resultado_final['vs_baseline'].items():
    target = resultado_final['targets'][comp]
    print('  (' + comp + ') ' + vs + ' | target ' + target)
readme_txt = 'PCES Benchmark Demo\nAutores: Will & Taipan\nScores: ' + json.dumps(resultado_final['scores'])
with open('/tmp/demo/README.txt', 'w') as f:
    f.write(readme_txt)
import zipfile
with zipfile.ZipFile('/tmp/demo/demo.zip', 'w') as zf:
    for fname in ['adj4.csv','adj8.csv','dataset.jsonl','figura_pces.png','README.txt']:
        zf.write('/tmp/demo/'+fname, fname)
print('demo.zip listo para DeepMind')